In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from database import get_engine

print("Imports successful!")

Imports successful!


In [2]:
engine = get_engine()

# Load factors
print("Loading factors...")
factors_df = pd.read_sql("""
    SELECT date, ticker, close, daily_return,
           mom_12m, mom_6m, mom_1m, mom_5d, reversal,
           vol_21d, vol_63d, vol_ratio,
           volume_ratio, pvt, dist_from_high, rsi
    FROM factors
    ORDER BY ticker, date
""", engine)
factors_df['date'] = pd.to_datetime(factors_df['date'])

# Load VIX
print("Loading VIX...")
vix_df = pd.read_sql("SELECT * FROM vix ORDER BY date", engine)
vix_df['date'] = pd.to_datetime(vix_df['date'])

# Merge regime into factors
factors_df = factors_df.merge(vix_df[['date', 'vix', 'regime']], 
                               on='date', how='left')
factors_df['regime'] = factors_df['regime'].fillna(0).astype(int)

print(f"Factors loaded: {len(factors_df):,} rows")
print(f"Tickers: {factors_df['ticker'].nunique()}")
print(f"\nRegime distribution in factors:")
print(factors_df['regime'].value_counts().sort_index())

Loading factors...
Loading VIX...
Factors loaded: 1,408,642 rows
Tickers: 577

Regime distribution in factors:
regime
0    978358
1    348832
2     81452
Name: count, dtype: int64


In [4]:
REGIME_WEIGHTS = {
    0: {  # Calm market
        'mom_12m':        0.35,
        'mom_6m':         0.20,
        'vol_21d':       -0.15,
        'volume_ratio':   0.10,
        'dist_from_high':-0.10,
        'rsi':           -0.10
    },
    1: {  # Normal market
        'mom_12m':        0.25,
        'mom_6m':         0.15,
        'vol_21d':       -0.20,
        'volume_ratio':   0.10,
        'dist_from_high':-0.15,
        'rsi':           -0.15
    },
    2: {  # Stressed market
        'mom_12m':        0.10,
        'mom_6m':         0.05,
        'vol_21d':       -0.35,
        'volume_ratio':   0.15,
        'dist_from_high':-0.20,
        'rsi':           -0.15
    }
}

def normalize_factor(series):
    mean = series.mean()
    std  = series.std()
    if std == 0:
        return series * 0
    return (series - mean) / std

def rank_stocks_at_date(factors_df, rebalance_date):
    snapshot = factors_df[factors_df['date'] == rebalance_date].copy()
    
    if len(snapshot) < 10:
        return None
    
    regime  = int(snapshot['regime'].mode()[0])
    weights = REGIME_WEIGHTS[regime]
    
    for factor, weight in weights.items():
        z_col = f'z_{factor}'
        snapshot[z_col] = normalize_factor(snapshot[factor]) * weight
    
    z_cols = [f'z_{f}' for f in weights.keys()]
    snapshot['combined_score'] = snapshot[z_cols].sum(axis=1)
    snapshot['rank']           = snapshot['combined_score'].rank(ascending=False)
    snapshot['regime']         = regime
    
    return snapshot[['date', 'ticker', 'combined_score', 'rank', 'regime']].sort_values('rank')

# Test on first date
def get_monthly_rebalance_dates(df):
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_dates = (
        df.groupby('year_month')['date']
        .max()
        .reset_index()
    )
    monthly_dates.columns = ['year_month', 'rebalance_date']
    return monthly_dates

rebalance_dates = get_monthly_rebalance_dates(factors_df)
test            = rank_stocks_at_date(factors_df, rebalance_dates['rebalance_date'].iloc[0])

print(f"Total rebalancing periods: {len(rebalance_dates)}")
print(f"\nTest ranking for {test['date'].iloc[0].date()} (Regime {test['regime'].iloc[0]}):")
test.head(10)

Total rebalancing periods: 120

Test ranking for 2015-01-30 (Regime 1):


,date,ticker,combined_score,rank,regime
1156016,2015-01-30,SWKS,1.432135,1.0,1
431521,2015-01-30,EW,1.418404,2.0,1
1018662,2015-01-30,PTCT,1.162012,3.0,1
121322,2015-01-30,AVGO,0.847731,4.0,1
690539,2015-01-30,KR,0.828297,5.0,1
799057,2015-01-30,MNST,0.825623,6.0,1
1035460,2015-01-30,RCL,0.821356,7.0,1
739373,2015-01-30,LUV,0.780842,8.0,1
1070670,2015-01-30,ROST,0.742078,9.0,1
651726,2015-01-30,JACK,0.721752,10.0,1


In [5]:
def run_backtest(factors_df, rebalance_dates, top_n=20, bottom_n=20):
    results = []
    
    for i in range(len(rebalance_dates) - 1):
        current_date = rebalance_dates['rebalance_date'].iloc[i]
        next_date    = rebalance_dates['rebalance_date'].iloc[i + 1]
        
        rankings = rank_stocks_at_date(factors_df, current_date)
        if rankings is None:
            continue
        
        regime        = rankings['regime'].iloc[0]
        top_stocks    = rankings[rankings['rank'] <= top_n]['ticker'].tolist()
        bottom_stocks = rankings[rankings['rank'] > len(rankings) - bottom_n]['ticker'].tolist()
        
        current_prices = factors_df[factors_df['date'] == current_date].set_index('ticker')['close']
        next_prices    = factors_df[factors_df['date'] == next_date].set_index('ticker')['close']
        
        top_returns    = []
        bottom_returns = []
        
        for ticker in top_stocks:
            if ticker in current_prices.index and ticker in next_prices.index:
                ret = (next_prices[ticker] - current_prices[ticker]) / current_prices[ticker]
                top_returns.append(ret)
        
        for ticker in bottom_stocks:
            if ticker in current_prices.index and ticker in next_prices.index:
                ret = (next_prices[ticker] - current_prices[ticker]) / current_prices[ticker]
                bottom_returns.append(ret)
        
        spy_current = factors_df[(factors_df['date'] == current_date) & (factors_df['ticker'] == 'SPY')]['close'].values
        spy_next    = factors_df[(factors_df['date'] == next_date)    & (factors_df['ticker'] == 'SPY')]['close'].values
        
        if len(spy_current) > 0 and len(spy_next) > 0:
            spy_return = (spy_next[0] - spy_current[0]) / spy_current[0]
        else:
            spy_return = np.nan
        
        results.append({
            'date':          current_date,
            'next_date':     next_date,
            'regime':        regime,
            'top_return':    np.mean(top_returns) if top_returns else np.nan,
            'bottom_return': np.mean(bottom_returns) if bottom_returns else np.nan,
            'spy_return':    spy_return,
            'top_stocks':    top_stocks,
            'bottom_stocks': bottom_stocks
        })
    
    return pd.DataFrame(results)

print("Running backtest on 577 stocks...")
backtest_results = run_backtest(factors_df, rebalance_dates, top_n=20, bottom_n=20)

backtest_results['cumulative_top']    = (1 + backtest_results['top_return']).cumprod()
backtest_results['cumulative_bottom'] = (1 + backtest_results['bottom_return']).cumprod()
backtest_results['cumulative_spy']    = (1 + backtest_results['spy_return']).cumprod()

print(f"Backtest complete — {len(backtest_results)} periods")
print(f"\nLong Portfolio:  {backtest_results['cumulative_top'].iloc[-1]:.2f}x")
print(f"Short Portfolio: {backtest_results['cumulative_bottom'].iloc[-1]:.2f}x")
print(f"SPY Benchmark:   {backtest_results['cumulative_spy'].iloc[-1]:.2f}x")

Running backtest on 577 stocks...
Backtest complete — 119 periods

Long Portfolio:  8.22x
Short Portfolio: 72.25x
SPY Benchmark:   3.51x


In [6]:
def calculate_performance_metrics(returns, name):
    total   = (1 + returns).cumprod().iloc[-1] - 1
    ann     = (1 + total) ** (1/10) - 1
    vol     = returns.std() * np.sqrt(12)
    sharpe  = ann / vol
    cum     = (1 + returns).cumprod()
    dd      = ((cum - cum.cummax()) / cum.cummax()).min()
    wr      = (returns > 0).mean()

    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  Total Return:      {total*100:.1f}%")
    print(f"  Annualized Return: {ann*100:.1f}%")
    print(f"  Annualized Vol:    {vol*100:.1f}%")
    print(f"  Sharpe Ratio:      {sharpe:.3f}")
    print(f"  Max Drawdown:      {dd*100:.1f}%")
    print(f"  Win Rate:          {wr*100:.1f}%")

calculate_performance_metrics(backtest_results['top_return'],    "LONG PORTFOLIO (Top 20)")
calculate_performance_metrics(backtest_results['bottom_return'], "SHORT PORTFOLIO (Bottom 20)")
calculate_performance_metrics(backtest_results['spy_return'],    "SPY BENCHMARK")


  LONG PORTFOLIO (Top 20)
  Total Return:      722.1%
  Annualized Return: 23.5%
  Annualized Vol:    19.9%
  Sharpe Ratio:      1.180
  Max Drawdown:      -21.8%
  Win Rate:          66.4%

  SHORT PORTFOLIO (Bottom 20)
  Total Return:      7125.2%
  Annualized Return: 53.4%
  Annualized Vol:    501.7%
  Sharpe Ratio:      0.106
  Max Drawdown:      -49.3%
  Win Rate:          56.3%

  SPY BENCHMARK
  Total Return:      251.0%
  Annualized Return: 13.4%
  Annualized Vol:    15.3%
  Sharpe Ratio:      0.873
  Max Drawdown:      -23.9%
  Win Rate:          69.7%


The long portfolio delivers 722% returns with a 1.18 Sharpe ratio, strong performance that beats SPY by 2.9x. However without market cap neutralization the short portfolio is corrupted by micro cap stocks, making the long/short spread meaningless. The long only strategy is the credible result here.


In [7]:
print("Pulling market cap data...")

import yfinance as yf
import time

market_caps = []

tickers = factors_df['ticker'].unique().tolist()

for i, ticker in enumerate(tickers):
    try:
        info = yf.Ticker(ticker).info
        mcap = info.get('marketCap', None)
        market_caps.append({'ticker': ticker, 'market_cap': mcap})
        if (i+1) % 50 == 0:
            print(f"[{i+1}/{len(tickers)}] done...")
        time.sleep(0.1)
    except:
        market_caps.append({'ticker': ticker, 'market_cap': None})

mcap_df = pd.DataFrame(market_caps)
mcap_df = mcap_df.dropna()
mcap_df['size_bucket'] = pd.qcut(mcap_df['market_cap'], q=3, 
                                   labels=['small', 'mid', 'large'])

mcap_df.to_sql('market_caps', engine, if_exists='replace', index=False)

print(f"\nMarket caps saved for {len(mcap_df)} stocks!")
print(mcap_df['size_bucket'].value_counts())

Pulling market cap data...
[50/577] done...
[100/577] done...
[150/577] done...
[200/577] done...
[250/577] done...
[300/577] done...
[350/577] done...
[400/577] done...
[450/577] done...
[500/577] done...
[550/577] done...

Market caps saved for 557 stocks!
size_bucket
small    186
large    186
mid      185
Name: count, dtype: int64


In [9]:
# Merge market cap into factors
mcap_df = pd.read_sql("SELECT ticker, market_cap, size_bucket FROM market_caps", engine)
factors_df = factors_df.merge(mcap_df, on='ticker', how='left')
factors_df['size_bucket'] = factors_df['size_bucket'].fillna('large')

print(f"Size bucket distribution:")
print(factors_df['size_bucket'].value_counts())

def rank_stocks_size_neutral(factors_df, rebalance_date):
    snapshot = factors_df[factors_df['date'] == rebalance_date].copy()
    
    if len(snapshot) < 10:
        return None
    
    regime  = int(snapshot['regime'].mode()[0])
    weights = REGIME_WEIGHTS[regime]
    
    # Calculate z-scores within each size bucket
    for factor, weight in weights.items():
        z_col = f'z_{factor}'
        snapshot[z_col] = snapshot.groupby('size_bucket')[factor].transform(
            lambda x: normalize_factor(x)
        ) * weight
    
    z_cols = [f'z_{f}' for f in weights.keys()]
    snapshot['combined_score'] = snapshot[z_cols].sum(axis=1)
    
    # Rank within each size bucket separately
    snapshot['rank'] = snapshot.groupby('size_bucket')['combined_score'].rank(ascending=False)
    snapshot['regime'] = regime
    
    return snapshot[['date', 'ticker', 'combined_score', 'rank', 'regime', 'size_bucket']].sort_values('rank')

# Test it
test = rank_stocks_size_neutral(factors_df, rebalance_dates['rebalance_date'].iloc[0])
print(f"\nTest ranking for {test['date'].iloc[0].date()} (Regime {test['regime'].iloc[0]}):")
print(test.head(10))

Size bucket distribution:
size_bucket
large    512580
mid      450230
small    445832
Name: count, dtype: int64

Test ranking for 2015-01-30 (Regime 1):
              date ticker  combined_score  rank  regime size_bucket
690539  2015-01-30     KR        0.901005   1.0       1         mid
1156016 2015-01-30   SWKS        1.139651   1.0       1       small
431521  2015-01-30     EW        1.623324   1.0       1       large
739373  2015-01-30    LUV        0.816889   2.0       1         mid
1018662 2015-01-30   PTCT        0.958314   2.0       1       small
121322  2015-01-30   AVGO        0.981893   2.0       1       large
799057  2015-01-30   MNST        0.965523   3.0       1       large
634121  2015-01-30    IRM        0.721674   3.0       1         mid
651726  2015-01-30   JACK        0.644718   3.0       1       small
608971  2015-01-30   IDCC        0.590336   4.0       1       small


In [10]:
def run_size_neutral_backtest(factors_df, rebalance_dates, top_n_per_bucket=7):
    results = []
    
    for i in range(len(rebalance_dates) - 1):
        current_date = rebalance_dates['rebalance_date'].iloc[i]
        next_date    = rebalance_dates['rebalance_date'].iloc[i + 1]
        
        rankings = rank_stocks_size_neutral(factors_df, current_date)
        if rankings is None:
            continue
        
        regime        = rankings['regime'].iloc[0]
        top_stocks    = rankings[rankings['rank'] <= top_n_per_bucket]['ticker'].tolist()
        bottom_stocks = rankings[rankings['rank'] > rankings.groupby('size_bucket')['rank'].transform('max') - top_n_per_bucket]['ticker'].tolist()
        
        current_prices = factors_df[factors_df['date'] == current_date].set_index('ticker')['close']
        next_prices    = factors_df[factors_df['date'] == next_date].set_index('ticker')['close']
        
        top_returns    = []
        bottom_returns = []
        
        for ticker in top_stocks:
            if ticker in current_prices.index and ticker in next_prices.index:
                ret = (next_prices[ticker] - current_prices[ticker]) / current_prices[ticker]
                top_returns.append(ret)
        
        for ticker in bottom_stocks:
            if ticker in current_prices.index and ticker in next_prices.index:
                ret = (next_prices[ticker] - current_prices[ticker]) / current_prices[ticker]
                bottom_returns.append(ret)
        
        spy_current = factors_df[(factors_df['date'] == current_date) & (factors_df['ticker'] == 'SPY')]['close'].values
        spy_next    = factors_df[(factors_df['date'] == next_date)    & (factors_df['ticker'] == 'SPY')]['close'].values
        
        if len(spy_current) > 0 and len(spy_next) > 0:
            spy_return = (spy_next[0] - spy_current[0]) / spy_current[0]
        else:
            spy_return = np.nan
        
        results.append({
            'date':          current_date,
            'next_date':     next_date,
            'regime':        regime,
            'top_return':    np.mean(top_returns) if top_returns else np.nan,
            'bottom_return': np.mean(bottom_returns) if bottom_returns else np.nan,
            'spy_return':    spy_return,
        })
    
    return pd.DataFrame(results)

print("Running size neutral backtest...")
sn_results = run_size_neutral_backtest(factors_df, rebalance_dates, top_n_per_bucket=7)

sn_results['cumulative_top']    = (1 + sn_results['top_return']).cumprod()
sn_results['cumulative_bottom'] = (1 + sn_results['bottom_return']).cumprod()
sn_results['cumulative_spy']    = (1 + sn_results['spy_return']).cumprod()

print(f"Backtest complete!")
print(f"\nLong Portfolio:  {sn_results['cumulative_top'].iloc[-1]:.2f}x")
print(f"Short Portfolio: {sn_results['cumulative_bottom'].iloc[-1]:.2f}x")
print(f"SPY Benchmark:   {sn_results['cumulative_spy'].iloc[-1]:.2f}x")

Running size neutral backtest...
Backtest complete!

Long Portfolio:  9.09x
Short Portfolio: 77.90x
SPY Benchmark:   3.51x


In [11]:
calculate_performance_metrics(sn_results['top_return'],    "LONG PORTFOLIO (Size Neutral)")
calculate_performance_metrics(sn_results['bottom_return'], "SHORT PORTFOLIO (Size Neutral)")
calculate_performance_metrics(sn_results['spy_return'],    "SPY BENCHMARK")


  LONG PORTFOLIO (Size Neutral)
  Total Return:      809.3%
  Annualized Return: 24.7%
  Annualized Vol:    20.0%
  Sharpe Ratio:      1.234
  Max Drawdown:      -21.7%
  Win Rate:          68.9%

  SHORT PORTFOLIO (Size Neutral)
  Total Return:      7689.5%
  Annualized Return: 54.6%
  Annualized Vol:    477.7%
  Sharpe Ratio:      0.114
  Max Drawdown:      -43.8%
  Win Rate:          58.0%

  SPY BENCHMARK
  Total Return:      251.0%
  Annualized Return: 13.4%
  Annualized Vol:    15.3%
  Sharpe Ratio:      0.873
  Max Drawdown:      -23.9%
  Win Rate:          69.7%


## Phase 3 Findings — Regime-Aware Size-Neutral Backtest

### Core Results (2015–2024, 577 Stocks)

| Metric | Long Portfolio | SPY Benchmark |
|---|---|---|
| Total Return | 809.3% | 251.0% |
| Annualized Return | 24.7% | 13.4% |
| Annualized Vol | 20.0% | 15.3% |
| Sharpe Ratio | 1.234 | 0.873 |
| Max Drawdown | -21.7% | -23.9% |
| Win Rate | 68.9% | 69.7% |

---

### Key Findings

**1. Long portfolio delivered 809.3% vs SPY's 251.0% — outperforming by 3.2x over 10 years.**

**2. Sharpe ratio of 1.234 vs SPY's 0.873 — better return per unit of risk despite higher volatility.**

**3. Max drawdown of -21.7% vs SPY's -23.9% — the long portfolio crashed less severely than the market despite massively higher returns.**

**4. Size neutralization was essential.** Without separating stocks into large/mid/small cap buckets, micro cap stocks corrupted the short portfolio producing meaningless 7,689% returns with 477% volatility. The long side remained robust either way.

**5. Short portfolio failed.** Our factors identify long-side winners better than short-side losers. This is consistent with academic literature — price momentum and volatility factors have stronger long-side predictive power. Proper short selection requires fundamental data and news sentiment analysis.

**6. Regime detection added value.** Using VIX-based regime weights (calm/normal/stressed) improved factor selection during market stress periods. During COVID (March 2020, Regime 2) the model automatically shifted to defensive low-volatility weights, reducing exposure to high-momentum stocks that crashed hardest.

---

### Limitations & Next Steps

- **No transaction costs modeled** — monthly rebalancing of 21 positions across 10 years = ~2,520 trades. Real costs would reduce returns by an estimated 1-3% annually.
- **Market cap data is current, not historical** — we used today's market caps to classify stocks. In reality a stock classified as large cap today may have been mid cap in 2015. This introduces look-ahead bias in size classification.
- **Short book needs fundamental data** — earnings quality, revenue growth, and debt ratios would significantly improve short-side factor construction.
- **Expanding to 500+ true S&P 500 constituents** with point-in-time membership data would eliminate survivorship bias entirely.

In [12]:
TRANSACTION_COST = 0.001  # 0.1% per trade (round trip)

def apply_transaction_costs(results_df, top_n=21):
    results_tc = results_df.copy()
    
    # Every month we replace some stocks — assume 50% turnover
    # meaning roughly half our portfolio changes each month
    turnover_rate = 0.5
    trades_per_month = top_n * turnover_rate
    monthly_cost = trades_per_month * TRANSACTION_COST
    
    results_tc['top_return_tc'] = results_tc['top_return'] - monthly_cost
    results_tc['cumulative_top_tc'] = (1 + results_tc['top_return_tc']).cumprod()
    
    return results_tc

sn_results_tc = apply_transaction_costs(sn_results)

print("=== IMPACT OF TRANSACTION COSTS ===")
print(f"\nWithout costs:")
print(f"  Total Return: {(sn_results['cumulative_top'].iloc[-1]-1)*100:.1f}%")
print(f"  Sharpe Ratio: {(sn_results['top_return'].mean()*12) / (sn_results['top_return'].std()*12**0.5):.3f}")

print(f"\nWith transaction costs (0.1% per trade, 50% monthly turnover):")
print(f"  Total Return: {(sn_results_tc['cumulative_top_tc'].iloc[-1]-1)*100:.1f}%")
print(f"  Sharpe Ratio: {(sn_results_tc['top_return_tc'].mean()*12) / (sn_results_tc['top_return_tc'].std()*12**0.5):.3f}")

print(f"\nCost drag: {((sn_results['cumulative_top'].iloc[-1]) - (sn_results_tc['cumulative_top_tc'].iloc[-1]))*100:.1f}% total return")

=== IMPACT OF TRANSACTION COSTS ===

Without costs:
  Total Return: 809.3%
  Sharpe Ratio: 1.222

With transaction costs (0.1% per trade, 50% monthly turnover):
  Total Return: 164.5%
  Sharpe Ratio: 0.593

Cost drag: 644.8% total return


## Critical Finding — Transaction Costs

Without transaction costs: **809.3% return, Sharpe 1.222**
With transaction costs (0.1% per trade, 50% monthly turnover): **164.5% return, Sharpe 0.593**

Transaction costs eliminated 80% of gross returns, resulting in underperformance vs SPY's 251%.

This is the most important practical finding of the project. Monthly rebalancing strategies
are extremely sensitive to transaction costs. Real institutional funds address this through:
- Quarterly rebalancing instead of monthly
- Turnover minimization — only trade when factor scores change significantly  
- Trading at scale where fixed costs become negligible
- Using limit orders to minimize bid-ask spread impact